Import Libraries

In [ ]:
# Parallelization
import os # Operating system interface
import tempfile
import multiprocessing as mp # Package for supporting spawning processes using APi similar to threading module
from multiprocessing import Pool # Allows pooling resources into subprocesses for making effective use of multiple cpus
from multiprocessing import cpu_count # Counts the number of cpu's in current system
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor # Class for parallelization that serves an updated version of pool

# MDAnalysis
import MDAnalysis as mda # Package for general simulation analysis
from MDAnalysis.analysis import rms # MDAnalysis function for analysis of root mean square quanities
from MDAnalysis.lib.distances import distance_array

# MDTraj 
import mdtraj as md # Python library for analyzing molecular dynamics trajectories

# Itertools
from itertools import product # Product function of python's itertools module that returns cartesian product of input iterables

# Hole2
from mdahole2.analysis.hole import HoleAnalysis # Function for pore interior analysis

# PyLipid
import pylipid # Python package for analyzing lipid interactions with membrane proteins from molecular dynamics simulations
from pylipid.util import get_traj_info # Gets trajectory info regarding atom/residue index and topologies
from pylipid.util import check_dir

# Plotting
# Color blind friendly color palettes for plotting
# Sunset spectrum (blue to tan to ruby): #364B9A #4A7BB7 #6EA6CD #98CAE1 #C2E4EF #EAECCC #FEDA8B #FDB366 #F67E4B #DD3D2D #A50026
# Viridis spectrum (gold to seafoam to violet): #FDE725 #5EC962 #21918C #3B528B #440154
# Magma spectrum (yellow to magenta to black): #FCFDBF #FC8961 #B73779 #51127C #000004
# Monochromatic teal spectrum (teal to aqua to black): #FEFFFF #DEF2F1 #3AAFA9 #2B7A78 #17252A
import matplotlib.pyplot as plt # Comprehensive library for creating visualizations in Python
import networkx as nx

# NGLView
import nglview as nv # Package for interactive visualization of proteins and trajectories

# NumPY
import numpy as np # Fundamental package for mathematical operations and numerical computations in Python

# SciPy
import scipy # Library that provides fundamental algorithms for optimization, integration, interpolation, and statistics
from scipy.ndimage import gaussian_filter1d # SciPy libtrary for gaussian filter (plot/image smoothing)
from scipy.spatial import ConvexHull # SciPy library that can take an index of points and find whether they form a convex hull
from scipy.spatial import Delaunay # SciPy library that supports Delaunay triangulation
from scipy.spatial import cKDTree # Scipy class that provides an index into k-dimensional points that finds nearest neighbors
from scipy.interpolate import interp1d # Scipy library for piecewise continuous interpolation
from scipy.interpolate import make_interp_spline
from scipy.signal import savgol_filter

# Pandas
import pandas as pd # Python library for data structure and data analysis

# Warnings
import warnings # Python core function for handling built-in exceptions
warnings.filterwarnings('ignore') # Ignore unnessary warnings from MDAnalysis

Analysis and Computing Class for RMSD/RMSF

Analysis and Computing Class for System Energies

In [ ]:
# Class for finding and storing information related to the energy of the protein. Requires GROMACS.
class IonChannelEA:
    StructureFileTypes = {'.psf','.pdb','.gro'} # Available structure file types
    TrajectoryFileTypes = {'.dcd','.xtc','.trr'} # Available trajectory file types
    EnergyFileTypes = {'.xvg','.csv','.log','.out','.mdout','.txt','.dat'} # Available energy file types

    # Function for initiating protein universes
    def __init__(self,Structure,Energies,Trajectory=None):
        if all(isinstance(Struct,str) for Struct in Structure): # If structure input is a string, assigns to structure
            Structures = Structure
        elif not isinstance(Structure,list): # Otherwise outputs an error for wrong input type
            raise TypeError("Structure must either be a string or a list of strings.")
        if Trajectory is None: # No trajectory file required, instead makes a blank structure for universe creation
            Trajectories = None * len(Structure)
        elif  all(isinstance(Traj,str) for Traj in Trajectory): # If trajectory exists and is a string/list of strings, creates trajectory object
            Trajectories = Trajectory
        elif not isinstance(Trajectory,list): # If provided and wrong file type, outputs error
            raise TypeError("Trajectory must be a string, a list of strings, or none.")
        if isinstance(Energies,str): # If energy input is a string, assigns to energy
            Energy = Energies
        elif not isinstance(Energies, list): # Otherwise outputs an error for wrong input type
            raise TypeError("Energy must either be a string or a list of strings.")
        if len(Structure) != len(Trajectory): # If there are not the same amount of structures and trajectories, produce error
            raise ValueError("The number of structures and trajectories must be the same.")
        elif len(Structure) != len(Energy): # If there are not the same amount of structures and energies, produce error
            raise ValueError("The number of structures and energy files must be the same.")
        for Struct in Structure: # Checks for every structure if in supported structure file types
            if os.path.splitext(Struct)[1].lower() not in self.StructureFileTypes:
                raise ValueError(f"File type {os.path.splitext(Struct)[1].lower()} is unsupported for structures.")
        for Traj in Trajectory: # Checks for every trajectory if in supported file types
            if Traj is not None:
                if os.path.splitext(Traj)[1].lower() not in self.TrajectoryFileTypes:
                    raise ValueError(f"File type {os.path.splitext(Traj)[1].lower()} is unsupported for trajectories.")
        for En in Energy: # Checks for every energy file if in supported file types
            if os.path.splitext(En)[1].lower() not in self.EnergyFileTypes:
                raise ValueError(f"File type {os.path.splitext(En)[1].lower()} is unsupported for energy files.")
        # Creates universe from structures and optionally and trajectory
        self.Universe = [mda.Universe(Struct,Traj) if Traj else mda.Universe(Struct) for Struct,Traj in zip(Structure,Trajectory)]
        self.Energy = Energy # Stores energy file for later reference

    # Function to find energy of the system over its trajectory from energy file
    def SystemEnergy(self,PotentialEnergyColumn,TotalEnergyColumn):
        self.PotentialEnergies = [] # List to store potential energies
        self.TotalEnergies = [] # List to store total energies
        for En in self.Energy:
            Ext = os.path.splitext(En)[1].lower() # Takes extension of energy file type
            # Can process for variety of output energy file types
            if Ext == '.xvg':
                Data = np.loadtxt(En,comments=['@','#']) # Takes data from .xvg file
                self.PotentialEnergies.append(Data[:,PotentialEnergyColumn]) # Stores potential energy
                self.TotalEnergies.append(Data[:,TotalEnergyColumn]) # Stores total energy
            elif Ext == '.csv' or '.dat':
                Data = pd.read_csv(En) # Takes data from .csv or .dat file
                self.PotentialEnergies.append(Data.iloc[:,PotentialEnergyColumn].values) # Stores potential energy
                self.TotalEnergies.append(Data.iloc[:,TotalEnergyColumn].values) # Stores total energy
            elif Ext == '.log' or Ext == '.txt':
                PotEnergyArray = [] # Creates an array for storing potential energies
                TotEnergyArray = [] # Creates an array for storing total energies
                with open(En) as File:
                    Reading = False # Tracker for reading line by line in file
                    for Line in File:
                        if 'Step' in Line and 'PotEng' in Line:
                            Reading = True # Updates reading when finding 
                            Headers = Line.split() # Splits line into columns
                            PotIDX = Headers.index('PotEng') # Stores index of potential energy
                            TotIDX = Headers.index('TotEng') if 'TotEng' in Headers else None # Stores index of potential energy
                            continue
                        if Reading:
                            if Line.strip() == '' or 'Loop time' in Line: 
                                break # Ends loop at end of documnet if empty
                            Columns = Line.split # Splits lines into columns
                            PotEnergyArray.append(float(Columns[PotIDX])) # Stores potential energy
                            TotEnergyArray.append(float(Columns[TotIDX]) if TotIDX is not None else 0.0) # Stores total energy
                self.PotentialEnergies.append(np.array(PotEnergyArray)) # Takes completed potential energy from lines
                self.TotalEnergies.append(np.array(TotEnergyArray)) # Takes completed total energy from lines
            elif Ext == '.out' or Ext == '.mdout':
                PotEnergyArray = [] # Creates an array for storing potential energies
                TotEnergyArray = [] # Creates an array for storing total energies
                with open(En) as File:
                    for Line in File:
                        if 'EPtot' in Line and 'EKtot' in Line and 'Etot' in Line:
                            Parts = line.split() # Stores parts of line
                            PotEnergyArray.append(float(Parts[2])) # Stores potential energy
                            TotEnergyArray.append(float(Parts[-1])) # Stores total energy
                self.PotentialEnergies.append(np.array(PotEnergyArray))  # Takes completed potential energy from lines
                self.TotalEnergies.append(np.array(TotEnergyArray))  # Takes completed total energy from lines     
            else:
                raise ValueError(f"Energy file format {Ext} is not supported.") # Raises error for wrong file type